# Reproduce the megalopa counting manuscript figures and tables

This CPU notebook reproduces Figures 1 to 10, Tables 1 to 11, and supplementary outputs from the study's retained prediction and measurement records. It verifies 46 numerical reference tables and includes counting accuracy, inference latency, sustained-operation monitoring, application timing, and whole-station power measurements.

The workflow recalculates analyses and generates publication outputs from saved evidence. It does not train models, perform new model inference, or compile deployment artifacts. Full training image datasets are not required.

## How to run

- **Reviewer ZIP:** upload all three source-package parts together when prompted. The notebook assembles them and verifies the package checksum.
- **GitHub:** load the source package from the repository after its files have been published. The notebook records the exact repository commit and verifies the package checksum. Public repository access does not require a token.
- Use a **CPU runtime** and select **Runtime > Run all**. Download the final results ZIP and execution receipt when the run finishes.

The results preserve the measurement boundaries described in the manuscript and supplement. Inference-call timing, application timing, and whole-station power are reported separately.

[Repository](https://github.com/AnawatBen/megalopa-counting-reproducibility) | [Open in Colab](https://colab.research.google.com/github/AnawatBen/megalopa-counting-reproducibility/blob/main/notebooks/Reproduce_Manuscript.ipynb)



In [ ]:
#@title 1. Choose inputs
SOURCE_MODE = "GitHub" #@param ["GitHub", "Reviewer ZIP"]
REPOSITORY = "AnawatBen/megalopa-counting-reproducibility" #@param {type:"string"}
REVISION = "main" #@param {type:"string"}
REVIEWER_ZIP = "/content/Phase1_Manuscript_Reproduction_Source.zip" #@param {type:"string"}
SAVE_TO_DRIVE = False #@param {type:"boolean"}
DOWNLOAD_RESULTS = False #@param {type:"boolean"}
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
if not Path('/content').is_dir():
    raise RuntimeError('Open this notebook in Google Colab. Local instructions are in README.md.')
EXPECTED_SOURCE_SHA256 = '46404a1617b67905fc9b74c3aa8311b1ba92716abc49305f04ee867a1ae7b878'
SOURCE_PARTS = ['Phase1_Manuscript_Reproduction_Source.zip.part01', 'Phase1_Manuscript_Reproduction_Source.zip.part02', 'Phase1_Manuscript_Reproduction_Source.zip.part03']
repository_commit = None
if SOURCE_MODE == 'GitHub':
    import tempfile, re
    if not re.fullmatch(r'[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+', REPOSITORY):
        raise ValueError('REPOSITORY must be owner/repository.')
    if not REVISION or REVISION.startswith('-'):
        raise ValueError('Enter a branch, tag or commit for REVISION.')
    # Public Git transport avoids shared Colab GitHub API rate limits.
    # Never read Colab secrets or request credentials.
    git_env = os.environ.copy()
    git_env['GIT_TERMINAL_PROMPT'] = '0'
    git_env['GIT_ASKPASS'] = '/bin/false'
    def public_git(args, cwd):
        result = subprocess.run(['git', '-c', 'credential.helper=', *args],
                                cwd=cwd, env=git_env, capture_output=True, text=True)
        if result.returncode:
            raise RuntimeError('Public GitHub download failed. No token is required. '
                               'Check the repository/revision and connection, then retry. '
                               'Alternatively use Reviewer ZIP. Details: ' + result.stderr.strip())
        return result.stdout.strip()
    print('Downloading public source from GitHub; no token required...')
    with tempfile.TemporaryDirectory(prefix='megalopa_public_') as tmp:
        public_git(['init', '--quiet'], tmp)
        public_git(['remote', 'add', 'origin', 'https://github.com/' + REPOSITORY + '.git'], tmp)
        public_git(['fetch', '--quiet', '--depth=1', 'origin', REVISION], tmp)
        repository_commit = public_git(['rev-parse', 'FETCH_HEAD'], tmp)
        public_git(['checkout', '--quiet', '--detach', repository_commit], tmp)
        data = b''.join((Path(tmp)/'reproduction'/part).read_bytes() for part in SOURCE_PARTS)
    if hashlib.sha256(data).hexdigest() != EXPECTED_SOURCE_SHA256:
        raise RuntimeError('Downloaded source archive differs from this notebook revision.')
    Path(REVIEWER_ZIP).write_bytes(data)
    print('Exact repository commit:', repository_commit)
if SOURCE_MODE in ('GitHub','Reviewer ZIP'):
    archive=Path(REVIEWER_ZIP)
    if not archive.exists():
        from google.colab import files
        print('Upload all three reproduction source .part files together, or the complete source ZIP.')
        uploads=files.upload()
        if all(n in uploads for n in SOURCE_PARTS):
            archive.write_bytes(b''.join(uploads[n] for n in SOURCE_PARTS))
        elif len(uploads)==1 and next(iter(uploads)).endswith('.zip'):
            archive=Path(next(iter(uploads))).resolve()
        else:
            raise RuntimeError('Upload all source parts together, or exactly one complete ZIP.')
    archive_hash=hashlib.sha256(archive.read_bytes()).hexdigest()
    if archive_hash != EXPECTED_SOURCE_SHA256: raise RuntimeError('Wrong source ZIP for this notebook version.')
    source=Path('/content/draft4_source_'+archive_hash[:16])
    source.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            target=(source/member.filename).resolve()
            if not target.is_relative_to(source.resolve()):
                raise RuntimeError('Unsafe archive member')
        z.extractall(source)
    print('Reviewer source ZIP SHA256:',archive_hash)
manifest_bytes=(source/'manifest.json').read_bytes()
fingerprint=hashlib.sha256(manifest_bytes).hexdigest()
manifest=json.loads(manifest_bytes)
local=Path('/content/draft4_unified_'+fingerprint[:16]);local.mkdir(exist_ok=True)
for rel,expected in manifest['files'].items():
    dst=(local/rel).resolve()
    if not dst.is_relative_to(local.resolve()):raise RuntimeError('Unsafe manifest path')
    dst.parent.mkdir(parents=True,exist_ok=True)
    if not dst.exists() or hashlib.sha256(dst.read_bytes()).hexdigest()!=expected:
        shutil.copy2(source/rel,dst)
    if hashlib.sha256(dst.read_bytes()).hexdigest()!=expected:raise RuntimeError('Source hash mismatch: '+rel)
(local/'manifest.json').write_bytes(manifest_bytes)
print('Verified unified source:',fingerprint)


In [ ]:
#@title 2. Prepare isolated CPU environment
venv=Path('/content/draft4_unified_env_'+fingerprint[:16]);python=venv/'bin/python'
if not python.exists():
    subprocess.run([sys.executable,'-m','venv','--without-pip',str(venv)],check=True)
requirements_hash=hashlib.sha256((local/'requirements.txt').read_bytes()).hexdigest()
marker=venv/'requirements.sha256'
if not marker.exists() or marker.read_text()!=requirements_hash:
    subprocess.run([sys.executable,'-m','pip','--python',str(python),'install','-r',str(local/'requirements.txt')],check=True)
    marker.write_text(requirements_hash)
print('CPU environment ready; no GPU required.')


In [ ]:
#@title 3. Recalculate, verify and build all outputs
output=local/'outputs'
env=os.environ.copy();env['MPLBACKEND']='Agg';env['EXECUTION_CONTEXT']='google_colab';env['OUTPUT_ROOT']=str(output)
with (local/'run.log').open('w') as log:
    proc=subprocess.Popen([str(python),'-u',str(local/'run_all.py')],cwd=local,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    for line in proc.stdout:
        print(line,end='',flush=True);log.write(line);log.flush()
    exit_code=proc.wait()
if exit_code:raise RuntimeError('Build failed. See output above and '+str(local/'run.log'))


In [ ]:
#@title 4. Preview figures and tables
from IPython.display import display,HTML,Image
status=json.loads((output/'output_status.json').read_text())
for item in status['catalog']:
    print(item['unit'],item['status'])
    p=output/item['preview']
    display(Image(filename=str(p),width=1000) if p.suffix=='.png' else HTML(p.read_text()))
print('Remaining evidence:')
for item in status['pending']:print(' -',item)


In [ ]:
#@title 5. Verify and save the single final export
receipt=json.loads((output/'UNIFIED_RUN_COMPLETE.json').read_text())
for rel,expected in receipt['files'].items():
    if hashlib.sha256((output/rel).read_bytes()).hexdigest()!=expected:raise RuntimeError('Output mismatch: '+rel)
saved=output
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    saved=Path('/content/drive/MyDrive/megalopa-edge-counting/outputs')/fingerprint[:20]
    shutil.copytree(output,saved,dirs_exist_ok=True)
    for rel,expected in receipt['files'].items():
        if hashlib.sha256((saved/rel).read_bytes()).hexdigest()!=expected:raise RuntimeError('Drive copy mismatch: '+rel)
archive=saved/'Phase1_Draft4_Unified_Publication_Outputs.zip'
print(json.dumps({'status':receipt['status'],'execution_context':receipt['execution_context'],'verified_numerical_tables':receipt['verified_numerical_tables'],'root':str(saved),'archive':str(archive),'archive_sha256':hashlib.sha256(archive.read_bytes()).hexdigest(),'model_execution':'none'},indent=2))
(saved/'REPOSITORY_SOURCE.json').write_text(json.dumps({'repository':REPOSITORY,'commit':repository_commit,'source_archive_sha256':EXPECTED_SOURCE_SHA256},indent=2))
print('Keep REPOSITORY_SOURCE.json alongside the ZIP. Keep the ZIP and receipts. Return this completion message for manuscript integration.')
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(archive))

if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(saved/"REPOSITORY_SOURCE.json"))
